In [71]:
# import kagglehub
# # let's authintificate ourselves with API token
# kagglehub.login()

In [72]:
# playground_series_s5e1_path = kagglehub.competition_download('playground-series-s5e1')
# playground_series_s5e1_path

In [73]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('./.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

././train.csv
././submission.csv
././test.csv
././playground-s5e1.ipynb
././catboost_info/catboost_training.json
././catboost_info/time_left.tsv
././catboost_info/learn_error.tsv
././catboost_info/learn/events.out.tfevents


In [74]:
import sklearn
print(sklearn.__version__)

1.6.0


In [75]:
df_train = pd.read_csv('train.csv')
df_train

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [76]:
df_train.drop('id', axis=1, inplace=True)

In [77]:
print(df_train.count())

date        230130
country     230130
store       230130
product     230130
num_sold    221259
dtype: int64


In [78]:
df_train['year'] = df_train['date'].apply(lambda x: int(x[:4]))
df_train['month'] = df_train['date'].apply(lambda x: int(x[5:7]))
df_train['day'] = df_train['date'].apply(lambda x: int(x[8:10]))
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['day of year'] = df_train['date'].dt.dayofyear
df_train['day of week'] = df_train['date'].dt.weekday
df_train['sin day of week'] = np.sin(2 * np.pi * df_train['day of week'] / 7)
df_train['cos day of week'] = np.cos(2 * np.pi * df_train['day of week'] / 7)
df_train['sin day of month'] = np.sin(2 * np.pi * df_train['day'] / 30)
df_train['cos day of month'] = np.cos(2 * np.pi * df_train['day'] / 30)
df_train['sin of month'] = np.sin(2 * np.pi * df_train['month'] / 12)
df_train['cos of month'] = np.cos(2 * np.pi * df_train['month'] / 12)
#df_train['seconds since 1970'] = df_train['date'].astype('int64') // 10 ** 9
df_train.drop('date', axis=1, inplace=True) #
df_train

,country,store,product,num_sold,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month
0,Canada,Discount Stickers,Holographic Goose,NaN,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
1,Canada,Discount Stickers,Kaggle,973.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
2,Canada,Discount Stickers,Kaggle Tiers,906.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
3,Canada,Discount Stickers,Kerneler,423.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
4,Canada,Discount Stickers,Kerneler Dark Mode,491.0,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,466.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230126,Singapore,Premium Sticker Mart,Kaggle,2907.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000
230128,Singapore,Premium Sticker Mart,Kerneler,1242.0,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000


In [79]:
# let's drop all rows with NaN values in the target column
df_train.dropna(subset=['num_sold'], inplace=True)

In [80]:
target = pd.DataFrame(df_train['num_sold'].astype('float64'))
target

,num_sold
1,973.0
2,906.0
3,423.0
4,491.0
5,300.0
...,...
230125,466.0
230126,2907.0
230127,2299.0
230128,1242.0


In [81]:
df_train.drop('num_sold', axis=1, inplace=True)
X = df_train.copy()

In [82]:
from catboost import CatBoostRegressor, Pool

train_pool = Pool(df_train, target, cat_features=['country', 'store', 'product'])

regressor = CatBoostRegressor(iterations=300, loss_function='MAPE', verbose=True)

In [83]:
#regressor.fit(train_pool)

In [84]:
df_test = pd.read_csv('test.csv')
ids = df_test['id']
df_test

,id,date,country,store,product
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle
2,230132,2017-01-01,Canada,Discount Stickers,Kaggle Tiers
3,230133,2017-01-01,Canada,Discount Stickers,Kerneler
4,230134,2017-01-01,Canada,Discount Stickers,Kerneler Dark Mode
...,...,...,...,...,...
98545,328675,2019-12-31,Singapore,Premium Sticker Mart,Holographic Goose
98546,328676,2019-12-31,Singapore,Premium Sticker Mart,Kaggle
98547,328677,2019-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers
98548,328678,2019-12-31,Singapore,Premium Sticker Mart,Kerneler


In [85]:
df_test.drop('id', axis=1, inplace=True)

In [86]:
df_test['year'] = df_test['date'].apply(lambda x: int(x[:4]))
df_test['month'] = df_test['date'].apply(lambda x: int(x[5:7]))
df_test['day'] = df_test['date'].apply(lambda x: int(x[8:10]))
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['day of year'] = df_test['date'].dt.dayofyear
df_test['day of week'] = df_test['date'].dt.weekday
df_test['sin day of week'] = np.sin(2 * np.pi * df_test['day of week'] / 7)
df_test['cos day of week'] = np.cos(2 * np.pi * df_test['day of week'] / 7)
df_test['sin day of month'] = np.sin(2 * np.pi * df_test['day'] / 30)
df_test['cos day of month'] = np.cos(2 * np.pi * df_test['day'] / 30)
df_test['sin of month'] = np.sin(2 * np.pi * df_test['month'] / 12)
df_test['cos of month'] = np.cos(2 * np.pi * df_test['month'] / 12)
#df_test['seconds since 1970'] = df_test['date'].astype('int64') // 10 ** 9
df_test.drop('date', axis=1, inplace=True) #
df_test

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month
0,Canada,Discount Stickers,Holographic Goose,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
1,Canada,Discount Stickers,Kaggle,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
2,Canada,Discount Stickers,Kaggle Tiers,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
3,Canada,Discount Stickers,Kerneler,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
4,Canada,Discount Stickers,Kerneler Dark Mode,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,Singapore,Premium Sticker Mart,Holographic Goose,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98546,Singapore,Premium Sticker Mart,Kaggle,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98547,Singapore,Premium Sticker Mart,Kaggle Tiers,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000
98548,Singapore,Premium Sticker Mart,Kerneler,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000


In [87]:
# pred = regressor.predict(df_test)

In [88]:
def season(month):
    match month:
        case 1 | 12: 
            return 'winter'
        case 2 | 3 | 4 | 5:
            return 'spring'
        case 6 | 7 | 8 :
            return 'summer'
        case _:
            return 'autumn'

df_train['season'] = df_train['month'].apply(season) 

In [89]:
df_train['day'] = df_train['day'].astype('category')
df_train['month'] = df_train['month'].astype('category')

df_train['sin day of year'] = np.sin(2 * np.pi * df_train['day of year'] / 365)
df_train['cos day of year'] = np.cos(2 * np.pi * df_train['day of year'] / 365)

df_train['day of year'] = df_train['day of year'].astype('category')

df_train

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
1,Canada,Discount Stickers,Kaggle,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
3,Canada,Discount Stickers,Kerneler,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
5,Canada,Stickers for Less,Holographic Goose,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230126,Singapore,Premium Sticker Mart,Kaggle,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230128,Singapore,Premium Sticker Mart,Kerneler,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852


In [90]:
df_train['day of week'] = df_train['day of week'].astype('category')
df_train

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
1,Canada,Discount Stickers,Kaggle,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
3,Canada,Discount Stickers,Kerneler,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
5,Canada,Stickers for Less,Holographic Goose,2010,1,1,1,4,-0.433884,-0.900969,0.207912,0.978148,5.000000e-01,0.866025,winter,0.017213,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230126,Singapore,Premium Sticker Mart,Kaggle,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852
230128,Singapore,Premium Sticker Mart,Kerneler,2016,12,31,366,5,-0.974928,-0.222521,0.207912,0.978148,-2.449294e-16,1.000000,winter,0.017213,0.999852


In [91]:
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

class DropNaNColumns(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        # Calculate the percentage of NaN values in each column
        nan_percentage = X.isna().mean()
        # Identify columns to drop
        self.columns_to_drop_ = nan_percentage[nan_percentage > self.threshold].index
        return self

    def transform(self, X):
        # Drop the identified columns
        return X.drop(columns=self.columns_to_drop_)


from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def preprocess_data(X, threshold=0.5, n_neighbors=1500, encode=True, impute=True):
    # Separate numerical and categorical columns
    numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_features = X.select_dtypes(include=['object', 'category']).columns


    feature_names = X.columns
    # Numerical feature pipeline: Imputation + Scaling
    if impute:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            # ('scaler', StandardScaler())  # Scale the numerical features
        ])
    else:
        numerical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            #('imputer', KNNImputer(n_neighbors=n_neighbors, weights='distance'))# SimpleImputer(strategy='mean')),#SimpleImputer()),#,
            # ('scaler', StandardScaler())  # Scale the numerical features
        ])


    # Categorical feature pipeline: Imputation + Encoding
    if encode:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            # ('imputer', SimpleImputer(strategy='constant', fill_value='.')),#SimpleImputer(strategy='constant', fill_value='.')), 
            ('encoder', OneHotEncoder(handle_unknown='ignore')) 
        ])
    else:
        categorical_pipeline = Pipeline(steps=[
            ('dropnan', DropNaNColumns(threshold=threshold)),
            ('imputer', SimpleImputer(strategy='constant', fill_value='.')), 
        ])

    # Define transformers
    preprocessor = Pipeline(steps=[
        ('transformer', ColumnTransformer(
        transformers=[
            ('num', numerical_pipeline, numerical_features),
            ('cat', categorical_pipeline, categorical_features)
        ],
        verbose=True,
        n_jobs=-1
    )),
    ('scaler', StandardScaler())
    ])

    return preprocessor

In [92]:
preprocessor = preprocess_data(X, threshold=0.8, encode=True, impute=False)
preprocessor

Pipeline(steps=[('transformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('dropnan',
                                                                   DropNaNColumns(threshold=0.8))]),
                                                  Index(['year', 'month', 'day', 'sin day of week', 'cos day of week',
       'sin day of month', 'cos day of month', 'sin of month', 'cos of month'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('dropnan',
                                                                   DropNaNColumns(threshold=0.8)),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['country', 'store', 'product'], dtype='object'))],
                                   verbose=True)),
                ('scaler', StandardScaler())])

In [93]:
from sklearn.neighbors import KNeighborsRegressor

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor(n_neighbors=20, weights='distance'))
])
model

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('transformer',
                                  ColumnTransformer(n_jobs=-1,
                                                    transformers=[('num',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8))]),
                                                                   Index(['year', 'month', 'day', 'sin day of week', 'cos day of week',
       'sin day of month', 'cos day of month', 'sin of month', 'cos of month'],
      dtype='object')),
                                                                  ('cat',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8)),
                                                                                   ('encoder',
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   Index(['country', 'store', 'product'], dtype='object'))],
                                                    verbose=True)),
                                 ('scaler', StandardScaler())])),
                ('regressor',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

In [94]:
from sklearn.model_selection import train_test_split

X_train_first, X_train_second, target_first, target_second = train_test_split(df_train, target, test_size=0.5, random_state=42)

In [95]:
model.fit(X_train_first, target_first)

[ColumnTransformer] ........... (1 of 2) Processing num, total=   0.0s
[ColumnTransformer] ........... (2 of 2) Processing cat, total=   0.1s


Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('transformer',
                                  ColumnTransformer(n_jobs=-1,
                                                    transformers=[('num',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8))]),
                                                                   Index(['year', 'month', 'day', 'sin day of week', 'cos day of week',
       'sin day of month', 'cos day of month', 'sin of month', 'cos of month'],
      dtype='object')),
                                                                  ('cat',
                                                                   Pipeline(steps=[('dropnan',
                                                                                    DropNaNColumns(threshold=0.8)),
                                                                                   ('encoder',
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   Index(['country', 'store', 'product'], dtype='object'))],
                                                    verbose=True)),
                                 ('scaler', StandardScaler())])),
                ('regressor',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

In [96]:
baseline = model.predict(X_train_second)

In [97]:
pool_w_baseline = Pool(X_train_second, target_second, cat_features=['country', 'store', 'product', 'day', 'month', 'day of year', 'day of week', 'season'], baseline=baseline) # [np.mean(target)]*len(target)

new_regressor = CatBoostRegressor(iterations=1000, loss_function='MAPE', verbose=True)

In [98]:
new_regressor.fit(pool_w_baseline)

0:	learn: 0.0808213	total: 74.1ms	remaining: 1m 13s
1:	learn: 0.0807348	total: 116ms	remaining: 57.9s
2:	learn: 0.0806584	total: 176ms	remaining: 58.4s
3:	learn: 0.0805768	total: 228ms	remaining: 56.8s
4:	learn: 0.0805013	total: 266ms	remaining: 52.9s
5:	learn: 0.0804266	total: 309ms	remaining: 51.2s
6:	learn: 0.0803546	total: 358ms	remaining: 50.7s
7:	learn: 0.0802885	total: 406ms	remaining: 50.3s
8:	learn: 0.0802213	total: 456ms	remaining: 50.2s
9:	learn: 0.0801565	total: 496ms	remaining: 49.1s
10:	learn: 0.0800896	total: 551ms	remaining: 49.5s
11:	learn: 0.0800221	total: 594ms	remaining: 48.9s
12:	learn: 0.0799660	total: 633ms	remaining: 48s
13:	learn: 0.0799131	total: 707ms	remaining: 49.8s
14:	learn: 0.0798596	total: 762ms	remaining: 50s
15:	learn: 0.0797996	total: 809ms	remaining: 49.8s
16:	learn: 0.0797456	total: 863ms	remaining: 49.9s
17:	learn: 0.0796911	total: 925ms	remaining: 50.5s
18:	learn: 0.0796455	total: 977ms	remaining: 50.5s
19:	learn: 0.0795918	total: 1.04s	remaining

38:	learn: 0.0788089	total: 2.08s	remaining: 51.2s
39:	learn: 0.0787797	total: 2.13s	remaining: 51s
40:	learn: 0.0787591	total: 2.18s	remaining: 51s
41:	learn: 0.0787344	total: 2.24s	remaining: 51s
42:	learn: 0.0787070	total: 2.3s	remaining: 51.3s
43:	learn: 0.0786731	total: 2.36s	remaining: 51.3s
44:	learn: 0.0786489	total: 2.4s	remaining: 51s
45:	learn: 0.0786256	total: 2.47s	remaining: 51.3s
46:	learn: 0.0786000	total: 2.55s	remaining: 51.7s
47:	learn: 0.0785767	total: 2.6s	remaining: 51.7s
48:	learn: 0.0785531	total: 2.66s	remaining: 51.6s
49:	learn: 0.0785049	total: 2.71s	remaining: 51.5s
50:	learn: 0.0784662	total: 2.78s	remaining: 51.8s
51:	learn: 0.0784252	total: 2.85s	remaining: 52s
52:	learn: 0.0784000	total: 2.92s	remaining: 52.1s
53:	learn: 0.0783688	total: 2.97s	remaining: 52s
54:	learn: 0.0783457	total: 3.04s	remaining: 52.2s
55:	learn: 0.0783276	total: 3.1s	remaining: 52.3s
56:	learn: 0.0783059	total: 3.16s	remaining: 52.3s
57:	learn: 0.0782826	total: 3.23s	remaining: 52

In [99]:
X_test = df_test.copy()

In [100]:
df_test['season'] = df_test['month'].apply(season)

In [101]:
df_test['day'] = df_test['day'].astype('category')
df_test['month'] = df_test['month'].astype('category')

df_test['sin day of year'] = np.sin(2 * np.pi * df_test['day of year'] / 365)
df_test['cos day of year'] = np.cos(2 * np.pi * df_test['day of year'] / 365)

df_test['day of year'] = df_test['day of year'].astype('category')
df_test['day of week'] = df_test['day of week'].astype('category')

df_test

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
0,Canada,Discount Stickers,Holographic Goose,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
1,Canada,Discount Stickers,Kaggle,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
3,Canada,Discount Stickers,Kerneler,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,Singapore,Premium Sticker Mart,Holographic Goose,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98546,Singapore,Premium Sticker Mart,Kaggle,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98547,Singapore,Premium Sticker Mart,Kaggle Tiers,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98548,Singapore,Premium Sticker Mart,Kerneler,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000


In [102]:
df_test

,country,store,product,year,month,day,day of year,day of week,sin day of week,cos day of week,sin day of month,cos day of month,sin of month,cos of month,season,sin day of year,cos day of year
0,Canada,Discount Stickers,Holographic Goose,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
1,Canada,Discount Stickers,Kaggle,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
2,Canada,Discount Stickers,Kaggle Tiers,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
3,Canada,Discount Stickers,Kerneler,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
4,Canada,Discount Stickers,Kerneler Dark Mode,2017,1,1,1,6,-0.781831,0.62349,0.207912,0.978148,5.000000e-01,0.866025,winter,1.721336e-02,0.999852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98545,Singapore,Premium Sticker Mart,Holographic Goose,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98546,Singapore,Premium Sticker Mart,Kaggle,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98547,Singapore,Premium Sticker Mart,Kaggle Tiers,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000
98548,Singapore,Premium Sticker Mart,Kerneler,2019,12,31,365,1,0.781831,0.62349,0.207912,0.978148,-2.449294e-16,1.000000,winter,6.432491e-16,1.000000


In [103]:
pred = new_regressor.predict(df_test) + model.predict(df_test)[:, 0] # np.array([np.mean(target)] * len(df_test))
pred

array([ 258.26523743,  756.60198345,  641.85906276, ..., 1739.10926594,
       1039.41293385, 1049.52213495])

In [104]:
df_pred = pd.DataFrame(pred, columns=['num_sold'])
df_pred['id'] = ids
df_pred = df_pred[['id', 'num_sold']]
df_pred

,id,num_sold
0,230130,258.265237
1,230131,756.601983
2,230132,641.859063
3,230133,317.136615
4,230134,440.812864
...,...,...
98545,328675,353.402514
98546,328676,2093.087457
98547,328677,1739.109266
98548,328678,1039.412934


In [105]:
df_pred.to_csv('submission.csv', index=False)